# Численное моделирование распространения сейсмических волн в двумерной среде MILEN SEM 2D. Часть третья.

## Глава V: SEG-Y модель для Tesseral

### Задача

Конвертировать декартову сетку материала модели с разломом в формат SEG-Y
для загрузки в программу Tesseral (конечно-разностное моделирование).

Аналог Главы II.1, но для модели с разломом (расширенная глубина до 2800 м).

Входные данные:
- `data/dev_3_2_fault_material.npz` — материал на декартовой сетке 5×5 м

Выходные данные:
- `data/dev_3_5_fault_Vp.sgy` — скорость P-волн (м/с)
- `data/dev_3_5_fault_Vs.sgy` — скорость S-волн (м/с)
- `data/dev_3_5_fault_Density.sgy` — плотность (г/см³)

In [16]:
import numpy as np
import segyio
from pathlib import Path

## Параметр угла разлома ##

In [17]:
# Угол разлома определяет суффикс входных/выходных файлов (должен совпадать с dev_3_1)
fault_angle_param = 10.0  # градусы
angle_suffix = f'_a{int(fault_angle_param)}'
print(f"Суффикс файлов: {angle_suffix}")

Суффикс файлов: _a10


## Загрузка данных ##

In [18]:
mat_data = np.load(f'data/dev_3_2_fault_material{angle_suffix}.npz')
material_grid = mat_data['material_grid']   # (2350, 560, 3) — E, nu, rho
coords_grid = mat_data['coords_grid']       # (2350, 560, 2) — x, y
fault_x = float(mat_data['fault_x'])
fault_throw = float(mat_data['fault_throw'])
model_bottom_depth = float(mat_data['model_bottom_depth'])

# Извлекаем свойства
E_grid = material_grid[:, :, 0]   # Модуль Юнга (Па)
nu_grid = material_grid[:, :, 1]  # Коэффициент Пуассона
rho_grid = material_grid[:, :, 2] # Плотность (кг/м³)

# Вычисляем скорости волн
Vp_grid = np.sqrt(E_grid / rho_grid * (1 - nu_grid) /
                  ((1 + nu_grid) * (1 - 2 * nu_grid)))
Vs_grid = np.sqrt(E_grid / (2 * rho_grid * (1 + nu_grid)))

# Плотность в г/см³ (стандарт для геофизики)
density_grid = rho_grid / 1000.0

# Параметры сетки
x_coords = coords_grid[:, 0, 0]
y_coords = coords_grid[0, :, 1]
nx, ny = len(x_coords), len(y_coords)
dx = x_coords[1] - x_coords[0]
dy = y_coords[1] - y_coords[0]

print(f"Материал загружен: {material_grid.shape}")
print(f"  Nx={nx}, Ny={ny}, dx={dx:.1f} м, dy={dy:.1f} м")
print(f"  X: {x_coords[0]:.1f} .. {x_coords[-1]:.1f} м")
print(f"  Y: {y_coords[0]:.1f} .. {y_coords[-1]:.1f} м")
print(f"\nДиапазоны:")
print(f"  Vp: {Vp_grid.min():.0f} .. {Vp_grid.max():.0f} м/с")
print(f"  Vs: {Vs_grid.min():.0f} .. {Vs_grid.max():.0f} м/с")
print(f"  ρ:  {density_grid.min():.3f} .. {density_grid.max():.3f} г/см³")

Материал загружен: (2350, 560, 3)
  Nx=2350, Ny=560, dx=5.0 м, dy=5.0 м
  X: 2.5 .. 11747.5 м
  Y: 2.5 .. 2797.5 м

Диапазоны:
  Vp: 1418 .. 5268 м/с
  Vs: 779 .. 2896 м/с
  ρ:  1.862 .. 2.578 г/см³


## Функция записи SEG-Y ##

In [19]:
def write_segy_2d(filename, data, dx, dy, x_origin=0.0):
    """
    Записывает 2D массив в формат SEG-Y (совместимый с Tesseral).

    Args:
        filename: путь к выходному файлу
        data: 2D массив [nx, ny] — трассы по X, сэмплы по Y (глубина)
        dx: шаг по X (м)
        dy: шаг по Y (м)
        x_origin: начало координат по X (м)
    """
    nx, ny = data.shape

    spec = segyio.spec()
    spec.format = 5  # IEEE float
    spec.samples = range(ny)
    spec.tracecount = nx
    spec.interval = int(dy * 1000)  # мкс (условно)

    with segyio.create(filename, spec) as f:
        f.bin = {
            segyio.BinField.JobID: 1,
            segyio.BinField.Samples: ny,
            segyio.BinField.Interval: spec.interval,
            segyio.BinField.Format: 5,
        }

        for i in range(nx):
            f.header[i] = {
                segyio.TraceField.TRACE_SEQUENCE_FILE: i + 1,
                segyio.TraceField.TRACE_SEQUENCE_LINE: i + 1,
                segyio.TraceField.INLINE_3D: i + 1,
                segyio.TraceField.CROSSLINE_3D: 1,
                segyio.TraceField.CDP_X: int(x_origin + i * dx),
                segyio.TraceField.CDP_Y: 0,
                segyio.TraceField.TRACE_SAMPLE_COUNT: ny,
                segyio.TraceField.TRACE_SAMPLE_INTERVAL: spec.interval,
            }
            f.trace[i] = data[i, :].astype(np.float32)

    file_size = Path(filename).stat().st_size / 1024 / 1024
    print(f"  {filename} — {nx}×{ny}, [{data.min():.1f}..{data.max():.1f}], {file_size:.1f} МБ")

## Запись SEG-Y файлов ##

In [20]:
print(f"\nЗапись SEG-Y файлов:")

write_segy_2d(f'data/dev_3_5_fault_Vp{angle_suffix}.sgy', Vp_grid, dx=dx, dy=dy, x_origin=x_coords[0])
write_segy_2d(f'data/dev_3_5_fault_Vs{angle_suffix}.sgy', Vs_grid, dx=dx, dy=dy, x_origin=x_coords[0])
write_segy_2d(f'data/dev_3_5_fault_Density{angle_suffix}.sgy', density_grid, dx=dx, dy=dy, x_origin=x_coords[0])


Запись SEG-Y файлов:
  data/dev_3_5_fault_Vp_a10.sgy — 2350×560, [1417.6..5267.8], 5.6 МБ
  data/dev_3_5_fault_Vs_a10.sgy — 2350×560, [778.8..2896.5], 5.6 МБ
  data/dev_3_5_fault_Density_a10.sgy — 2350×560, [1.9..2.6], 5.6 МБ


## Проверка записанных файлов ##

In [21]:
print(f"\nПроверка:")
for name, path in [('Vp', f'data/dev_3_5_fault_Vp{angle_suffix}.sgy'),
                   ('Vs', f'data/dev_3_5_fault_Vs{angle_suffix}.sgy'),
                   ('Density', f'data/dev_3_5_fault_Density{angle_suffix}.sgy')]:
    with segyio.open(path, 'r', ignore_geometry=True) as f:
        data_check = segyio.tools.collect(f.trace[:])
        header0 = f.header[0]
        print(f"  {name}: shape={data_check.shape}, "
              f"range=[{data_check.min():.2f}..{data_check.max():.2f}], "
              f"CDP_X[0]={header0[segyio.TraceField.CDP_X]}")


Проверка:
  Vp: shape=(2350, 560), range=[1417.55..5267.81], CDP_X[0]=2
  Vs: shape=(2350, 560), range=[778.80..2896.45], CDP_X[0]=2
  Density: shape=(2350, 560), range=[1.86..2.58], CDP_X[0]=2


## Визуализация ##

In [22]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

X, Y = np.meshgrid(x_coords, y_coords, indexing='ij')

for ax, data, title, label in zip(
    axes,
    [Vp_grid, Vs_grid, density_grid],
    ['Vp', 'Vs', 'Плотность'],
    ['м/с', 'м/с', 'г/см³']
):
    c = ax.pcolormesh(X, Y, data, cmap='rainbow', shading='auto')
    plt.colorbar(c, ax=ax, label=label)
    ax.axvline(fault_x, color='k', linewidth=1, linestyle='--')
    ax.set_xlabel('X (м)')
    ax.set_ylabel('Глубина (м)')
    ax.set_title(f'{title} (SEG-Y)')
    ax.set_ylim(model_bottom_depth, 0)

plt.tight_layout()
plt.savefig(f'img/dev_3_5_segy_overview{angle_suffix}.png', dpi=200, bbox_inches='tight')
plt.close()
print(f"\nСохранено: img/dev_3_5_segy_overview{angle_suffix}.png")


Сохранено: img/dev_3_5_segy_overview_a10.png


## Сводка ##

In [23]:
print(f"\n{'='*60}")
print(f"=== Сводка Главы V ===")
print(f"{'='*60}")
print(f"Модель: {nx}×{ny} (шаг {dx:.0f}×{dy:.0f} м)")
print(f"Глубина: 0 .. {model_bottom_depth:.0f} м")
print(f"Разлом: x={fault_x:.0f} м, throw={fault_throw:.0f} м")
print(f"\nФайлы SEG-Y:")
print(f"  data/dev_3_5_fault_Vp{angle_suffix}.sgy      — Vp [{Vp_grid.min():.0f}..{Vp_grid.max():.0f}] м/с")
print(f"  data/dev_3_5_fault_Vs{angle_suffix}.sgy      — Vs [{Vs_grid.min():.0f}..{Vs_grid.max():.0f}] м/с")
print(f"  data/dev_3_5_fault_Density{angle_suffix}.sgy — ρ  [{density_grid.min():.3f}..{density_grid.max():.3f}] г/см³")


=== Сводка Главы V ===
Модель: 2350×560 (шаг 5×5 м)
Глубина: 0 .. 2800 м
Разлом: x=5875 м, throw=50 м

Файлы SEG-Y:
  data/dev_3_5_fault_Vp_a10.sgy      — Vp [1418..5268] м/с
  data/dev_3_5_fault_Vs_a10.sgy      — Vs [779..2896] м/с
  data/dev_3_5_fault_Density_a10.sgy — ρ  [1.862..2.578] г/см³


## Выводы ##

Выполнена конвертация декартовой сетки материала модели с разломом в формат SEG-Y:

1. Загружен материал из Главы III.2 (E, ν, ρ на сетке 5×5 м)
2. Вычислены сейсмические скорости Vp и Vs
3. Плотность конвертирована из кг/м³ в г/см³
4. Записаны 3 файла SEG-Y (формат IEEE float, трассы по X, сэмплы по глубине)
5. Файлы совместимы с Tesseral для конечно-разностного моделирования

Формат идентичен Главе II.1 — расширена только область по глубине (2800 м вместо 2750 м).